<a href="https://colab.research.google.com/github/naman-0804/learning/blob/Langchain/llmapp_LCEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building LLM Prompt And StrOutput Parser Chain With LCEL

First, we'll load the `GROQ_API` key from Colab's secrets manager. Ensure you have added your GROQ API key to the secrets manager named `GROQ_API`.

In [ ]:
# Used to securely store your API key
from google.colab import userdata

GROQ_API=userdata.get('GROQ_API_KEY')

In [ ]:
#!pip install langchain_groq
#!pip install langchain_core

In [ ]:
from langchain_groq import ChatGroq
model=ChatGroq(model="llama-3.3-70b-versatile",groq_api_key=GROQ_API)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.13'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x7bd7f9d1c800>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7bd7f9d169c0>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [ ]:
from langchain_core.messages import HumanMessage,SystemMessage

In [ ]:
messages=[
    SystemMessage(content="You are a helpful assistant that translates English to French."),
    HumanMessage(content="I love programming.")
]
result=model.invoke(messages)
result

AIMessage(content="J'adore la programmation.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 50, 'total_tokens': 59, 'completion_time': 0.047077376, 'completion_tokens_details': None, 'prompt_time': 0.007970437, 'prompt_tokens_details': None, 'queue_time': 0.058380487, 'total_time': 0.055047813}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fff5e-13aa-7752-bb38-975e78bf311f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 50, 'output_tokens': 9, 'total_tokens': 59})

In [ ]:
from langchain_core.output_parsers import StrOutputParser
parser=StrOutputParser()
parser.invoke(result)

"J'adore la programmation."

In [ ]:
#Using LCEL - Chain the components
#Bypass the long method by chaining

In [ ]:
chain=model | parser
chain.invoke(messages) #message sent to model then to parser

"J'adore la programmation."

In [ ]:
#Primpt templates
from langchain_core.prompts import ChatPromptTemplate
generic_template="Translate the following into {language}:"
prompt=ChatPromptTemplate.from_messages(
    [("system",generic_template),("user","{text}")]
)
#Converting useless msg written by human to list of structured prompt

In [ ]:
result=prompt.invoke({"language":"French","text":"Hello"})

In [ ]:
result.to_messages()

[SystemMessage(content='Translate the following into French:', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})]

In [ ]:
chain=prompt|model|parser #prompt adds the gernic template to the question send to model then
chain.invoke({"language":"french","text":"hello"})

'Bonjour'

### Explanation of the LCEL Chain Flow: `prompt | model | parser`

Here is how the data flows through the chain step-by-step:

1.  **`prompt`**: This is the first step. It takes your input dictionary (e.g., `{"language": "french", "text": "hello"}`) and injects those values into the predefined template. It transforms the raw text into a structured list of messages (System and User messages) that the AI model expects.
2.  **`model`**: The structured messages from the prompt are passed directly into the LLM (ChatGroq). The model processes these instructions and returns a raw `AIMessage` object containing the translation and metadata.
3.  **`parser` (`StrOutputParser`)**: This is the final step. It receives the `AIMessage` from the model and 'strips away' all the metadata, extracting only the `content` string. This results in the clean text output like `'Bonjour'`.